<a href="https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content item, summarized over a fixed decision-point date (same grain established in `w02`). Source: `fact_content_daily_performance` (the warehouse daily fact table) — the starter CSV has no daily granularity and can't build this grain at all.

**Time windows, all relative to the decision point (e.g. 2026-03-31):**
- **Feature window:** trailing 90 days before the decision point (`2025-12-31` to `2026-03-30`) — used for volume-style features.
- **Trend-check window (30 vs 30 days):** the most recent 60 of those 90 days, split in half — matches FlyRank's own `trend_pct` convention (verified in `w02`).
- **Label window:** the 30 days *after* the decision point (`2026-03-31` to `2026-04-30`) — strictly future, never touched by any feature above.

The grain and window claims are verified with real queries in section 3, not just stated here.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Context** (grouping/filtering, never a feature) | `report_date`, `client_hash_id`, `content_hash_id`, `month`, `keyword_hash_id`, `url_hash_id`; `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`, `is_published`, `is_deleted`, `gsc_data_start`/`ga4_data_start` (from `dim_clients`) | IDs/dates are for joining and windowing only. The `*_available` flags gate whether a zero means "no activity" or "no tracking yet." `is_published`/`is_deleted` are filters. `gsc_data_start`/`ga4_data_start` decide which clients' windows are even valid (see section 3/4) — a filter, not a feature. |
| **Feature candidates, verified usable (daily facts)** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `gsc_sum_position` | GSC coverage is clean (42.1-42.5% zero, all explained by tracking gaps verified in section 3) — safe to build on directly. |
| **Feature candidates, usable but need the availability flag first** | `ga4_pageviews` (71.2% zero), `ga4_sessions` (71.3%), `ga4_users` (71.3%), `ga4_engaged_sessions` (94.7%), `ga4_total_engagement_sec` (87.4%), `sessions_organic` (86.5%), `sessions_direct` (79.7%) | GA4 tracking coverage is only 28.8% (verified in section 3) — a zero here can mean "no tracking" or "genuinely no activity," and only joining `ga4_data_available` per row tells them apart. |
| **Feature candidates, too sparse to trust alone (verified)** | `sessions_ai` (98.3% zero), `scroll_events` (85.2%), `sessions_referral` (97.7%), `sessions_social` (99.8%), `sessions_paid` (98.0%), `ai_chatgpt` (98.9%), `ai_perplexity` (99.7%), `ai_gemini` (99.5%) | Real data, but so few non-zero rows that a model would mostly be fitting noise if these carried real weight alone. |
| **Excluded — completely empty this window (verified)** | `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | 100.0% zero for every content item in this 90-day window — not sparse, literally no data at all in this slice. Worth re-checking against a different decision point before permanently ruling them out, but they carry zero information right now. |
| **Feature candidates, verified usable (from `dim_content`)** | `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent` (all 18.3-18.5% missing, matches "blank when no keyword data" per `docs/data-dictionary.md`), `content_type`, `word_count`/`char_count` (30.8% missing), `category_count` (0% missing), `keyword_char_count`/`keyword_token_count`/`url_char_count`, `content_created_date`/`content_updated_date` (convert to day-counts relative to the decision point, same as `content_age_days`/`days_since_last_update`) | Content metadata, static or slow-changing, knowable well before any decision point — legitimate features once missingness is handled with `has_keyword_data`-style flags rather than blind fills (same lesson as `word_count` in `w01`/`w02`). |
| **Feature candidate, sparse** | `backlinks` (53.0% missing) | Real signal when present, but missing for over half of items — needs a `has_backlink_data` flag, not a blind fill. |
| **Needs its own leakage audit before use (ML-05) — do not assume safe** | `last_optimized_date`, `optimization_eligible_date` (both 87.8% missing — far sparser than everything else, and only populate for a small minority) | The sparsity pattern plus the name strongly suggest these only populate when FlyRank's own system *acted* on a page — exactly the "product decision as a feature" trap from section 4 of the lane guide (`health_score`, `priority_score`, etc.). Not excluded outright, since it's unverified either way, but must not be used as a feature until ML-05 confirms what actually populates it and when. |
| **Excluded** | `provider_used`, `model_used` | Explicitly marked "not a model feature" in the starter CSV's own data dictionary — reveals internal content-generation tooling, not a performance signal. |
| **Excluded for this decision point (verified leakage risk)** | Every column in `fact_content_query_90d` (`impressions_90d`, `avg_position_last30`, `rare_query_count`, `rare_impressions_share`, etc.) | Its own window (verified in section 3: `2026-04-02` to `2026-06-30`) starts *inside* our label window (`2026-03-31` to `2026-04-30`) and runs past it. Using any of its columns as a feature for this decision point would leak the future into the past. This table is built for a decision point aligned with its own window — ours isn't one of them. |
| **Feature (prior-window only)** | `was_declining` / `prior_trend_pct` | Computed entirely from the 30-vs-30 trend-check window, strictly before the decision point — safe. |
| **Label / proxy** (never a feature) | `future_change_pct`, `future_decline`, `future_recovery`, `future_momentum` | All computed from the label window, strictly after the decision point — this is literally the thing being predicted. |
| **Excluded** | Any FlyRank product decision flag (`health_score`, `priority_score`, `action_type`, etc.) | Not shipped in this data at all — nothing to strip out, but noting it so it's clear why: using a product's own decision as a feature just teaches the model to copy that decision (`docs/ml-intern-dataset-and-lane-guide.md`, section 4). |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
%pip install -q duckdb huggingface_hub pandas python-dotenv

import os
import duckdb
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
local_files = [
    hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={m}/data_0.parquet",
        token=token,
    )
    for m in MONTHS
]

con = duckdb.connect()
file_list = ", ".join(f"'{f}'" for f in local_files)
REL = f"read_parquet([{file_list}])"
DECISION_DATE = "2026-03-31"

query = f"""
SELECT content_hash_id,
    COUNT(*) AS n_days,
    SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position,
    SUM(ga4_sessions) AS ga4_sessions, SUM(ga4_engaged_sessions) AS engaged_sessions,
    SUM(sessions_ai) AS ai_sessions, SUM(scroll_events) AS scroll_events,
    MIN(report_date) AS min_d, MAX(report_date) AS max_d
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df = con.sql(query).df()

print("Rows (unique content items in the 90-day feature window):", len(df))
print()

# Grain check: zero rows back means the grain holds (df is already GROUP BY
# content_hash_id, so this also double-checks pandas agrees with the SQL).
dupes = df["content_hash_id"].duplicated().sum()
print("Grain check -- duplicate content_hash_id rows:", dupes, "(0 = grain holds)")
print()

print("Missingness -- share of content items with ZERO of this metric in the 90-day window:")
for col in ["impressions", "clicks", "ga4_sessions", "engaged_sessions", "ai_sessions", "scroll_events"]:
    print(f"  {col}: {(df[col].fillna(0) == 0).mean():.1%}")
print(f"  avg_position (no GSC position data at all): {df['avg_position'].isna().mean():.1%}")
print()

print("Window actually pulled:", df["min_d"].min(), "to", df["max_d"].max())
print()
print("First few rows:")
df.head()

Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rows (unique content items in the 90-day feature window): 349205

Grain check -- duplicate content_hash_id rows: 0 (0 = grain holds)

Missingness -- share of content items with ZERO of this metric in the 90-day window:
  impressions: 42.1%
  clicks: 75.0%
  ga4_sessions: 71.3%
  engaged_sessions: 94.7%
  ai_sessions: 98.3%


  scroll_events: 85.2%
  avg_position (no GSC position data at all): 42.5%

Window actually pulled: 2025-12-31 00:00:00 to 2026-03-30 00:00:00

First few rows:


,content_hash_id,n_days,impressions,clicks,avg_position,ga4_sessions,engaged_sessions,ai_sessions,scroll_events,min_d,max_d
0,content_543e7c98eeb71167,88,520.0,0.0,18.841082,0.0,0.0,0.0,0.0,2025-12-31,2026-03-30
1,content_713522a1da1d37e2,59,26.0,0.0,44.555556,0.0,0.0,0.0,0.0,2025-12-31,2026-03-30
2,content_0d80ac02899326ab,84,271.0,0.0,9.650961,0.0,0.0,0.0,0.0,2025-12-31,2026-03-30
3,content_cbdb19a539f4095a,90,1183.0,0.0,32.907333,0.0,0.0,0.0,0.0,2025-12-31,2026-03-30
4,content_830f2ccb62778a7c,90,1324.0,1.0,51.764006,0.0,0.0,0.0,0.0,2025-12-31,2026-03-30


**GA4-availability check.** The 94.7% zero rate for `engaged_sessions` above blends two very different things: pages that genuinely got no engagement, and pages GA4 simply wasn't tracking yet. This splits them apart using `ga4_data_available`.

In [2]:
query_ga4 = f"""
SELECT content_hash_id,
    BOOL_OR(ga4_data_available) AS ga4_ever_available,
    SUM(ga4_engaged_sessions) AS engaged_sessions
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df_ga4 = con.sql(query_ga4).df()

tracked = df_ga4[df_ga4["ga4_ever_available"] == True]
untracked = df_ga4[df_ga4["ga4_ever_available"] == False]
undetermined = df_ga4["ga4_ever_available"].isna().sum()

print("GA4 tracking coverage over the 90-day window:")
print(f"  never tracked (ga4_ever_available = False): {len(untracked)} ({len(untracked)/len(df_ga4):.1%})")
print(f"  tracked at some point (= True):              {len(tracked)} ({len(tracked)/len(df_ga4):.1%})")
print(f"  undetermined (flag itself missing):          {undetermined} ({undetermined/len(df_ga4):.1%})")
print()
print("engaged_sessions == 0, split by tracking status:")
print(f"  among TRACKED items   -> real zero engagement: {(tracked['engaged_sessions'].fillna(0)==0).mean():.1%}")
print(f"  among UNTRACKED items -> fake zero, just not measured yet: {(untracked['engaged_sessions'].fillna(0)==0).mean():.1%}")

GA4 tracking coverage over the 90-day window:
  never tracked (ga4_ever_available = False): 180572 (51.7%)
  tracked at some point (= True):              100590 (28.8%)
  undetermined (flag itself missing):          68043 (19.5%)

engaged_sessions == 0, split by tracking status:
  among TRACKED items   -> real zero engagement: 81.5%
  among UNTRACKED items -> fake zero, just not measured yet: 100.0%


**GSC-availability check on `clicks`.** Same question for `gsc_clicks` (75.0% zero overall): is that a tracking gap like GA4, or something else? `gsc_data_available` is cleanly populated here (no undetermined rows), so the split is unambiguous.

In [3]:
query_gsc = f"""
SELECT content_hash_id,
    BOOL_OR(gsc_data_available) AS gsc_ever_available,
    SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df_gsc = con.sql(query_gsc).df()

tracked_gsc = df_gsc[df_gsc["gsc_ever_available"] == True]
untracked_gsc = df_gsc[df_gsc["gsc_ever_available"] == False]

print("GSC tracking coverage over the 90-day window:")
print(f"  never tracked: {len(untracked_gsc)} ({len(untracked_gsc)/len(df_gsc):.1%})")
print(f"  tracked at some point: {len(tracked_gsc)} ({len(tracked_gsc)/len(df_gsc):.1%})")
print()
print("clicks == 0, split by tracking status:")
print(f"  among TRACKED items   -> real zero-click pages: {(tracked_gsc['clicks'].fillna(0)==0).mean():.1%}")
print(f"  among UNTRACKED items -> fake zero, just not measured yet: {(untracked_gsc['clicks'].fillna(0)==0).mean():.1%}")
print()
print("Unlike GA4: no undetermined rows here, and the tracked-item zero rate (56.8%) is a")
print("legitimate SEO finding, not a data gap -- most tracked, impression-getting pages")
print("still never get an actual click. gsc_impressions/gsc_clicks look clean enough to")
print("build on directly; ga4_engaged_sessions needs the availability flag joined in first.")

GSC tracking coverage over the 90-day window:
  never tracked: 147132 (42.1%)
  tracked at some point: 202073 (57.9%)

clicks == 0, split by tracking status:
  among TRACKED items   -> real zero-click pages: 56.8%
  among UNTRACKED items -> fake zero, just not measured yet: 100.0%

Unlike GA4: no undetermined rows here, and the tracked-item zero rate (56.8%) is a
legitimate SEO finding, not a data gap -- most tracked, impression-getting pages
still never get an actual click. gsc_impressions/gsc_clicks look clean enough to
build on directly; ga4_engaged_sessions needs the availability flag joined in first.


**The rest of the feature-candidate list, actually checked.** Section 2 lists 12+ candidate columns, but only a handful were verified above -- per this skill's own rule ("a claim without a query is a guess"), the remaining ones need the same treatment before they can honestly stay "candidates."

In [4]:
remaining_cols = [
    "gsc_sum_position", "ga4_pageviews", "ga4_users", "ga4_total_engagement_sec",
    "sessions_organic", "sessions_direct", "sessions_referral", "sessions_social", "sessions_paid",
    "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other",
]
sums = ", ".join(f"SUM({c}) AS {c}" for c in remaining_cols)

query_remaining = f"""
SELECT content_hash_id, {sums}
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df_remaining = con.sql(query_remaining).df()

print("Missingness for the rest of the candidate list, n =", len(df_remaining))
for c in remaining_cols:
    zero_rate = (df_remaining[c].fillna(0) == 0).mean()
    flag = "  <- COMPLETELY EMPTY this window" if zero_rate == 1.0 else ""
    print(f"  {c}: {zero_rate:.1%} zero{flag}")

Missingness for the rest of the candidate list, n = 349205
  gsc_sum_position: 42.5% zero
  ga4_pageviews: 71.2% zero
  ga4_users: 71.3% zero
  ga4_total_engagement_sec: 87.4% zero
  sessions_organic: 86.5% zero
  sessions_direct: 79.7% zero
  sessions_referral: 97.7% zero
  sessions_social: 99.8% zero
  sessions_paid: 98.0% zero
  ai_chatgpt: 98.9% zero
  ai_perplexity: 99.7% zero
  ai_gemini: 99.5% zero
  ai_copilot: 100.0% zero
  ai_claude: 100.0% zero
  ai_meta: 100.0% zero  <- COMPLETELY EMPTY this window
  ai_other: 100.0% zero  <- COMPLETELY EMPTY this window


**`dim_content` — content metadata, not yet touched at all.** Everything checked above only comes from `fact_content_daily_performance` (daily performance numbers). Things like search volume, intent, and word count live in a separate table, `dim_content`, which this notebook hadn't joined in until now.

In [5]:
dim_content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="dim_content.parquet", token=token,
)

query_dim = f"""
WITH pop AS (
    SELECT DISTINCT content_hash_id FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
      AND report_date < DATE '{DECISION_DATE}'
)
SELECT p.content_hash_id, d.*
FROM pop p LEFT JOIN read_parquet('{dim_content_file}') d USING (content_hash_id)
"""
df_dim = con.sql(query_dim).df()

print("Join check -- population:", len(df_dim),
      "| matched to dim_content:", df_dim["content_type"].notna().sum(),
      f"({df_dim['content_type'].notna().mean():.1%})")
print()
print("Missingness on the new candidate fields:")
for col in ["search_volume", "competition", "cpc", "main_intent", "word_count", "char_count",
            "backlinks", "category_count", "last_optimized_date", "optimization_eligible_date"]:
    print(f"  {col}: {df_dim[col].isna().mean():.1%} missing")
print()
print("is_published:", df_dim["is_published"].value_counts(dropna=False).to_dict())
print("is_deleted:  ", df_dim["is_deleted"].value_counts(dropna=False).to_dict())

Join check -- population: 349205 | matched to dim_content: 349205 (100.0%)

Missingness on the new candidate fields:
  search_volume: 18.5% missing
  competition: 18.5% missing
  cpc: 18.5% missing
  main_intent: 18.3% missing
  word_count: 30.8% missing
  char_count: 30.8% missing
  backlinks: 53.0% missing
  category_count: 0.0% missing
  last_optimized_date: 87.8% missing
  optimization_eligible_date: 87.8% missing

is_published: {True: 328069, False: 21136}
is_deleted:   {False: 331885, True: 17320}


**The other two warehouse tables — checked for the first time.** `dim_clients` (per-client history coverage) and `fact_content_query_90d` (query-mix features) hadn't been touched at all until now.

In [6]:
# dim_clients -- does our 90-day prior window actually have full coverage per client?
clients_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                filename="dim_clients.parquet", token=token)
df_clients = con.sql(f"SELECT * FROM read_parquet('{clients_file}')").df()

prior_window_start = "2025-12-31"
gap_clients = (df_clients["gsc_data_start"] > prior_window_start).sum()
print(f"dim_clients: {len(df_clients)} clients total")
print(f"  gsc_data_start AFTER our prior-window start ({prior_window_start}):",
      f"{gap_clients} of {len(df_clients)} ({gap_clients/len(df_clients):.1%})")
print("  -> for these clients, part of our 90-day window predates their tracking,")
print("     so a zero there is a coverage gap, not a real signal. Confirms the")
print("     'unbalanced panel' limit below with an exact number instead of a guess.")
print()

# fact_content_query_90d -- is its window even compatible with our decision point,
# or would using it leak future data into our features?
query_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                              filename="fact_content_query_90d.parquet", token=token)
w = con.sql(f"SELECT MIN(window_start) AS min_start, MAX(window_end) AS max_end FROM read_parquet('{query_file}')").df()
print("fact_content_query_90d actual window range:", w["min_start"][0], "to", w["max_end"][0])
print("Our decision point: 2026-03-31 | label window: 2026-03-31 to 2026-04-30")
print("-> the query table's window starts 2026-04-02 -- already INSIDE our label")
print("   window, and runs past it. Using this table's columns as features for THIS")
print("   decision point would leak future data. Excluded for this lane/decision")
print("   point -- it's built for a decision point aligned with its own window,")
print("   which ours isn't. Exactly the leakage warning the lane guide gives for")
print("   this table, now confirmed against our actual dates rather than assumed.")

dim_clients: 104 clients total
  gsc_data_start AFTER our prior-window start (2025-12-31): 27 of 104 (26.0%)
  -> for these clients, part of our 90-day window predates their tracking,
     so a zero there is a coverage gap, not a real signal. Confirms the
     'unbalanced panel' limit below with an exact number instead of a guess.



fact_content_query_90d actual window range: 2026-04-02 00:00:00 to 2026-06-30 00:00:00
Our decision point: 2026-03-31 | label window: 2026-03-31 to 2026-04-30
-> the query table's window starts 2026-04-02 -- already INSIDE our label
   window, and runs past it. Using this table's columns as features for THIS
   decision point would leak future data. Excluded for this lane/decision
   point -- it's built for a decision point aligned with its own window,
   which ours isn't. Exactly the leakage warning the lane guide gives for
   this table, now confirmed against our actual dates rather than assumed.


**Last two columns, closing out the full sweep.** `client_has_ga4` (a structural, account-level flag in the daily fact table) and `dim_clients`' own completeness -- do all 104 clients even have a client dimension record?

In [7]:
q_flag = f"""
SELECT content_hash_id, BOOL_OR(client_has_ga4) AS has_ga4
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df_flag = con.sql(q_flag).df()
print("client_has_ga4 (structural account flag, not a tracking-start check):")
print(" ", df_flag["has_ga4"].value_counts(dropna=False).to_dict())
print("  -> ~25% of content items belong to clients that never had GA4 access at")
print("     all, distinct from the ga4_data_available gap already covered: this is")
print("     permanent, not a 'hasn't started yet' situation.")
print()

print("dim_clients completeness:")
print(" ", df_clients["access_profile"].value_counts(dropna=False).to_dict())
orphans = (df_clients["access_profile"] == "source_only_missing_client_dimension").sum()
print(f"  -> {orphans} of {len(df_clients)} clients ({orphans/len(df_clients):.1%}) have NO client")
print("     dimension record at all -- they appear in the fact/content tables but")
print("     dim_clients has nothing for them. A genuine data-quality edge case,")
print("     not something to silently fillna over.")

client_has_ga4 (structural account flag, not a tracking-start check):
  {True: 260539, False: 88666}
  -> ~25% of content items belong to clients that never had GA4 access at
     all, distinct from the ga4_data_available gap already covered: this is
     permanent, not a 'hasn't started yet' situation.

dim_clients completeness:
  {'gsc_and_ga4': 53, 'no_search_or_analytics_access': 26, 'gsc_only': 14, 'source_only_missing_client_dimension': 10, 'ga4_only': 1}
  -> 10 of 104 clients (9.6%) have NO client
     dimension record at all -- they appear in the fact/content tables but
     dim_clients has nothing for them. A genuine data-quality edge case,
     not something to silently fillna over.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell you:**

- **No causality.** Even a strong flag never proves a refresh *caused* a recovery — that needs an actual experiment, not this data (`DATA_USE.md`).
- **Unbalanced panel — verified, exact number.** 27 of 104 clients (26.0%) have a `gsc_data_start` *after* our prior-window start (2025-12-31) — for these clients, part of our 90-day window predates their tracking entirely, so a zero there is a coverage gap, not a real signal. (Manifest-level: only 9 of 70 clients have 12+ months of history overall.)
- **GA4 engagement — verified, not just theoretical.** Only 28.8% of content items had GA4 tracking active for even one day of the 90-day window; 51.7% had none at all, and another 19.5% have an undetermined status (the `ga4_data_available` flag itself missing on every row). The naive 94.7% zero rate for `engaged_sessions` blends real behavior with tracking gaps — split apart, **100% of untracked items show a fake zero** (nothing to measure), while **81.5% of genuinely tracked items still show a real zero** (most pages just don't get deep engagement, which is itself a legitimate finding, not a data gap). A further ~25% of content items belong to clients whose `client_has_ga4` is `False` entirely — a *permanent* lack of access, not just "hasn't started tracking yet."
- **GSC clicks — a cleaner story.** Checked the same way, `gsc_data_available` has no undetermined rows: 42.1% of items were never tracked (matching the 42.1% impression-zero rate exactly), 57.9% were tracked. Among tracked items, 56.8% still show zero clicks — but that's a legitimate SEO finding (most impressions never convert to a click), not a tracking artifact. `gsc_impressions`/`gsc_clicks` look reliable enough to use directly; `ga4_engaged_sessions` needs the availability flag joined in first before trusting a zero.
- **Extremely sparse columns.** `ai_sessions` is zero for 98.3% of items, `scroll_events` for 85.2% — matches the lane guide's warning that AI-referral rows are rare. Any model or claim resting heavily on these needs to say so explicitly, and probably shouldn't stand alone.
- **No ranking data for ~42% of items.** `avg_position` is missing for 42.5% of content items in the 90-day window (no impressions logged at all) — same "0 doesn't mean rank 1" trap as the starter CSV, just showing up as a true NULL here instead of a sentinel zero.
- **`fact_content_query_90d` is unusable for this decision point — verified.** Its actual window (`2026-04-02` to `2026-06-30`) starts *inside* our label window and runs well past it. Any of its columns used as a feature here would leak the future into the past. It's excluded entirely for this lane's current decision point, not because the table is bad, but because its window and ours don't line up.
- **`dim_clients` itself is incomplete.** 10 of 104 clients (9.6%) have no client dimension record at all (`access_profile = source_only_missing_client_dimension`) — they appear in the fact/content tables but have nothing here, so any client-level feature built from `dim_clients` will be missing for these clients specifically, not randomly.
- **Single decision point so far.** Everything verified in `w02`/`w03` uses one fixed decision date (2026-03-31). A real capstone needs multiple decision points (walk-forward style) to know these numbers aren't an artifact of that one month — not yet built.
- **Host reliability.** The Hugging Face remote read was intermittently flaky during this work (`ZSTD Decompression failure` on certain files) — reproducing these exact numbers may require a retry or two.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.